# Formatting Statistics

This notebook demonstrates how to create publication-ready formatted statistics using format_statistics().

In [1]:
import pandas as pd
import numpy as np
from pyMyriad import AnalysisTree, simple_table, gt_table
from pyMyriad.tabular import format_statistics

In [2]:
# Create sample data and run analysis
np.random.seed(42)
df = pd.DataFrame(
    {
        "Gender": np.random.choice(["M", "F"], 200),
        "Country": np.random.choice(["US", "UK", "Canada"], 200),
        "Income": np.random.normal(50000, 15000, 200),
    }
)

atree = (
    AnalysisTree()
    .split_by("df.Gender")
    .split_by("df.Country")
    .analyze_by(
        mean=lambda df: np.mean(df.Income),
        sd=lambda df: np.std(df.Income),
        n=lambda df: len(df),
        ci_lower=lambda df: (
            np.mean(df.Income) - 1.96 * np.std(df.Income) / np.sqrt(len(df))
        ),
        ci_upper=lambda df: (
            np.mean(df.Income) + 1.96 * np.std(df.Income) / np.sqrt(len(df))
        ),
    )
)

result = atree.run(df)

## View Raw Statistics

First, look at the unformatted statistics.

In [3]:
print(simple_table(result).head(10))

  _Level_0 _Level_1 _Level_2 _Level_3 Statistic         Value
5   Gender        F  Country   Canada      mean  54375.460432
5                                            sd  14215.930754
5                                             n            31
5                                      ci_lower   49371.07762
5                                      ci_upper  59379.843244
7                                  UK      mean  50510.396621
7                                            sd  16970.685709
7                                             n            37
7                                      ci_lower  45042.068074
7                                      ci_upper  55978.725168


## Basic Formatting: Mean ± SD

Create a formatted string combining mean and standard deviation.

In [5]:
# Apply formatting to create "mean ± sd" statistic
format_statistics(result, inplace=True, mean_sd="{mean:.1f} ± {sd:.1f}")

# Display the new formatted statistic
print(simple_table(result).head(10))

  _Level_0 _Level_1 _Level_2 _Level_3 Statistic              Value
5   Gender        F  Country   Canada      mean       54375.460432
5                                            sd       14215.930754
5                                             n                 31
5                                      ci_lower        49371.07762
5                                      ci_upper       59379.843244
5                                       mean_sd  54375.5 ± 14215.9
7                                  UK      mean       50510.396621
7                                            sd       16970.685709
7                                             n                 37
7                                      ci_lower       45042.068074


## Format Confidence Intervals

Create formatted CI strings.

In [6]:
# Create a new result for this example
result2 = atree.run(df)

# Format confidence interval
format_statistics(result2, inplace=True, ci_95="({ci_lower:.0f}, {ci_upper:.0f})")

print(simple_table(result2).head(10))

  _Level_0 _Level_1 _Level_2 _Level_3 Statistic           Value
5   Gender        F  Country   Canada      mean    54375.460432
5                                            sd    14215.930754
5                                             n              31
5                                      ci_lower     49371.07762
5                                      ci_upper    59379.843244
5                                         ci_95  (49371, 59380)
7                                  UK      mean    50510.396621
7                                            sd    16970.685709
7                                             n              37
7                                      ci_lower    45042.068074


## Multi-Round Formatting

Apply formatting in multiple steps to build complex formatted statistics.

In [9]:
# Create a new result
result3 = atree.run(df)

# Round 1: Create mean ± SD
format_statistics(result3, inplace=True, mean_sd="{mean:.1f} ± {sd:.1f}")

# Round 2: Create CI range
format_statistics(result3, inplace=True, ci_range="({ci_lower:.0f}, {ci_upper:.0f})")

# Round 3: Combine into full summary
format_statistics(result3, inplace=True, full_summary="{mean_sd}, 95% CI: {ci_range}")

simple_table(result3).head(8)

,_Level_0,_Level_1,_Level_2,_Level_3,Statistic,Value
5,Gender,F,Country,Canada,mean,54375.460432
5,,,,,sd,14215.930754
5,,,,,n,31
5,,,,,ci_lower,49371.07762
5,,,,,ci_upper,59379.843244
5,,,,,mean_sd,54375.5 ± 14215.9
5,,,,,ci_range,"(49371, 59380)"
5,,,,,full_summary,"54375.5 ± 14215.9, 95% CI: (49371, 59380)"


## Include Sample Size in Formatting

Create formatted statistics with n (sample size).

In [10]:
# Create a new result
result4 = atree.run(df)

# Format with sample size
format_statistics(result4, inplace=True, mean_sd_n="{mean:.0f} ± {sd:.0f} (n={n})")

simple_table(result4)

,_Level_0,_Level_1,_Level_2,_Level_3,Statistic,Value
5,Gender,F,Country,Canada,mean,54375.460432
5,,,,,sd,14215.930754
5,,,,,n,31
5,,,,,ci_lower,49371.07762
5,,,,,ci_upper,59379.843244
5,,,,,mean_sd_n,54375 ± 14216 (n=31)
7,,,,UK,mean,50510.396621
7,,,,,sd,16970.685709
7,,,,,n,37
7,,,,,ci_lower,45042.068074


## Publication-Ready Table

Create a final publication-quality table with formatted statistics.

In [11]:
# Create a new result
result = atree.run(df)

result = format_statistics(
    result, remove_original=True, formatted="{mean:.0f} ± {sd:.0f} (n={n})"
)

gt_table(result, pivot_statistics=False, by="df.Country")

GT(_tbl_data=pivot_lvl _Level_0 _Level_1  Statistic                Canada  \
0           Gender        F   ci_lower           49371.07762   
1                             ci_upper          59379.843244   
2                            formatted  54375 ± 14216 (n=31)   
3                         M   ci_lower          41182.307224   
4                             ci_upper          50891.200981   
5                            formatted  46037 ± 14653 (n=35)   

pivot_lvl                    UK                    US  
0                  45042.068074          52644.473286  
1                  55978.725168          60564.342784  
2          50510 ± 16971 (n=37)  56604 ± 11429 (n=32)  
3                  45926.325039           43540.15915  
4                  55019.956913          51530.577691  
5          50473 ± 11365 (n=24)  47535 ± 13052 (n=41)  , _body=<great_tables._gt_data.Body object at 0x11699b2f0>, _boxhead=Boxhead([ColInfo(var='_Level_0', type=<ColInfoTypeEnum.default: 1>, column_label='_Level_0', column_align='left', column_width=None), ColInfo(var='_Level_1', type=<ColInfoTypeEnum.default: 1>, column_label='_Level_1', column_align='left', column_width=None), ColInfo(var='Statistic', type=<ColInfoTypeEnum.default: 1>, column_label='Statistic', column_align='left', column_width=None), ColInfo(var='Canada', type=<ColInfoTypeEnum.default: 1>, column_label='Canada', column_align='left', column_width=None), ColInfo(var='UK', type=<ColInfoTypeEnum.default: 1>, column_label='UK', column_align='left', column_width=None), ColInfo(var='US', type=<ColInfoTypeEnum.default: 1>, column_label='US', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x116b04bc0>, _spanners=Spanners([SpannerInfo(spanner_id='Hierarchy', spanner_level=0, spanner_label='Hierarchy', spanner_units=None, spanner_pattern=None, vars=['_Level_0', '_Level_1'], built=None)]), _heading=Heading(title='Analysis Summary', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x116802fc0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1168013d0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x116801640>, _formats=[], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='12px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, 